# 06 — Model Experiments (Phase 2: Feature Engineering)

## 1. Tujuan

Phase 1 memakai seluruh variabel asli. Hasil *mean score decrease* menunjukkan bahwa hanya sebagian kecil variabel yang berpengaruh terhadap *churn*, sedangkan sisanya hampir nol atau negatif. Sebagian model juga bahkan ada yg mengalami overfitting. Phase 2 bertujuan memperbaiki representasi data dengan dua langkah:

1. Membuat fitur baru yang merangkum informasi yang tersebar di banyak kolom.
2. Memilih hanya satu fitur untuk setiap informasi yang sama, agar model tidak menerima fitur yang saling tumpang tindih.

## 2. Temuan dari Phase 1

| Kelompok | Variabel | Keterangan |
|---|---|---|
| Berpengaruh kuat | `Contract`, `tenure`, `InternetService` | Berada di posisi teratas pada hampir semua model |
| Berpengaruh sedang | `MonthlyCharges`, `TotalCharges`, `PaymentMethod` | Nilainya bervariasi antar model |
| Berpengaruh lemah | `OnlineSecurity`, `TechSupport`, `PaperlessBilling`, `StreamingTV`, `OnlineBackup` | Nilainya kecil jika berdiri sendiri |
| Hampir tidak berpengaruh | `gender`, `SeniorCitizen`, `Partner`, `Dependents`, `PhoneService`, `MultipleLines`, `DeviceProtection` | Nol atau negatif pada sebagian besar model |

## 3. Fitur Baru dan Alasannya

| Fitur baru | Cara pembuatan | Alasan |
|---|---|---|
| `tenure_bin` | `tenure` dikelompokkan menjadi 0-12, 13-24, 25-48, dan 49+ bulan | Risiko *churn* umumnya tertinggi pada tahun pertama dan menurun setelahnya. Pola ini tidak linear, sehingga pengelompokan membantu model seperti Logistic Regression, SVM, dan KNN menangkapnya. |
| `charge_diff` | `TotalCharges - tenure × MonthlyCharges` | `TotalCharges` hampir sama dengan `tenure × MonthlyCharges`. Selisihnya menunjukkan apakah tagihan pelanggan berubah selama berlangganan, yaitu informasi yang tidak dimiliki `TotalCharges` maupun `MonthlyCharges` secara terpisah. |
| `n_protection` | Jumlah layanan proteksi yang digunakan (`OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`) | Masing-masing layanan lemah jika berdiri sendiri. Jumlahnya menggambarkan seberapa terikat pelanggan dengan layanan pendukung. |
| `n_streaming` | Jumlah layanan streaming yang digunakan (`StreamingTV`, `StreamingMovies`) | Dipisah dari layanan proteksi karena sifat layanannya berbeda, sehingga pengaruhnya terhadap *churn* bisa berlawanan arah. |
| `is_echeck` | 1 jika `PaymentMethod` adalah *Electronic check* | Fokus pada metode pembayaran yang diduga paling berkaitan dengan *churn* (dapat diverifikasi pada hasil EDA), tanpa memecah menjadi empat kategori. |
| `has_family` | 1 jika pelanggan memiliki `Partner` atau `Dependents` | Kedua variabel lemah jika berdiri sendiri, sehingga digabung menjadi satu indikator status keluarga. |

## 4. Prinsip Pemilihan Fitur: Satu Informasi, Satu Fitur

Fitur yang membawa informasi yang sama tidak dimasukkan bersamaan. Kolinearitas dapat membagi dan mengaburkan nilai pentingnya, terutama pada KNN, SVM, dan Logistic Regression.

| Informasi | Dipakai | Tidak dipakai | Alasan |
|---|---|---|---|
| Lama berlangganan | `tenure_bin` | `tenure` | Keduanya menjelaskan lama berlangganan |
| Biaya | `MonthlyCharges`, `charge_diff` | `TotalCharges` | `TotalCharges` dapat dihitung dari `tenure` dan `MonthlyCharges` |
| Layanan tambahan | `n_protection`, `n_streaming` | Kolom layanan asli | Informasinya sudah terwakili jumlah layanan |
| Pembayaran | `is_echeck` | `PaymentMethod` | `is_echeck` adalah ringkasan dari `PaymentMethod` |
| Keluarga | `has_family` | `Partner`, `Dependents` | Keduanya sudah terwakili `has_family` |

Variabel `gender`, `SeniorCitizen`, `PhoneService`, dan `MultipleLines` tidak dipakai karena tidak berpengaruh pada hasil phase 1.

## 5. Fitur Final yang Digunakan (10 fitur)

| Asli | Hasil feature engineering |
|---|---|
| `Contract` (ordinal) | `tenure_bin` |
| `InternetService` | `charge_diff` |
| `MonthlyCharges` | `n_protection` |
| `PaperlessBilling` | `n_streaming` |
| | `is_echeck` |
| | `has_family` |

## 6. Catatan Metodologi

- Seluruh fitur dihitung per baris tanpa memakai statistik dari keseluruhan data, sehingga tidak menimbulkan *data leakage* meskipun dibuat sebelum pembagian data.
- Data dibagi ulang dengan `random_state` yang sama dengan phase 1 agar perbandingan skor adil.
- Hasil phase 2 dibandingkan dengan phase 1 menggunakan CV F1 dan Test F1. Fitur yang tidak menaikkan skor akan dievaluasi lagi di phase 3 melalui *mean score decrease*.